# Tetris RL Model Zoo — Colab

VSCode Colab extension에서 이 노트북을 열고 위에서부터 실행하세요.

이 노트북은 PPO/DQN/DDQN/CBMPI/REINFORCE/A2C/n-step AC/CEM/MuZero-style 중 하나를 골라 학습하고, 결과를 `model/bots/*.onnx`로 export합니다. 로컬 Mac mini에서는 학습을 돌리지 않고, Colab에서 만든 `.onnx`만 내려받아 사용합니다.

## 1. 설정

`REPO_URL`을 본인 fork로 바꾸고, `ALGO`를 원하는 알고리즘으로 바꾸세요.

가능한 값: `ppo`, `ppo_sparse`, `dqn`, `ddqn`, `cbmpi`, `cbmpi_value`, `reinforce`, `a2c`, `nstep_ac`, `cem`, `muzero`

In [ ]:
REPO_URL = 'https://github.com/<USER>/Tetris-Multiplayer-RL.git'
REPO_DIR = '/content/Tetris-Multiplayer-RL'

ALGO = 'ddqn'
RUN_NAME = f'aria_{ALGO}'

# Smoke 값으로 먼저 검증하고, 잘 돌면 아래 TRAIN_PRESET을 'long'으로 바꾸세요.
TRAIN_PRESET = 'smoke'  # 'smoke' or 'long'

print('algo    :', ALGO)
print('run name:', RUN_NAME)
print('preset  :', TRAIN_PRESET)

## 2. 저장소 준비

In [ ]:
import os
import subprocess
from pathlib import Path

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
subprocess.run(['git', 'pull'], check=True)
print('cwd:', os.getcwd())

## 3. 의존성 설치 + `tetris_py` 빌드

Colab 런타임이 초기화될 때마다 한 번 실행합니다.

In [ ]:
!apt-get install -qq cmake g++ python3-dev > /dev/null
!pip install -q -r python/requirements-colab.txt

# Stale native modules are the usual cause of:
# AttributeError: 'tetris_py.SimGame' object has no attribute 'clone'
!rm -rf build
!rm -f python/sim/tetris_py*.so

import glob
import shutil
import subprocess

PYBIND11_DIR = subprocess.check_output(['python', '-m', 'pybind11', '--cmakedir']).decode().strip()
print('pybind11 cmake dir:', PYBIND11_DIR)

!cmake -S . -B build \
  -DCMAKE_BUILD_TYPE=Release \
  -DTETRIS_BUILD_GAME=OFF \
  -DTETRIS_BUILD_PY=ON \
  -DTETRIS_BUILD_TEST=ON \
  -Dpybind11_DIR=$PYBIND11_DIR
!cmake --build build -j --target tetris_py sim_hash_dump

Path('python/sim').mkdir(parents=True, exist_ok=True)
for so in glob.glob('build/tetris_py*.so'):
    print('copying', so, '-> python/sim/')
    shutil.copy(so, 'python/sim/')

## 4. 스모크 테스트

`SimGame.clone()`까지 확인합니다. CBMPI가 이 clone을 사용합니다.

In [ ]:
import sys
sys.path.insert(0, 'python')

from sim import SimGame
from common.env import TetrisPlacementEnv

g = SimGame(seed=42)
if not hasattr(g, 'clone'):
    raise RuntimeError('SimGame.clone() is missing. Re-run the build cell after git pull; delete build/ and python/sim/tetris_py*.so if needed.')
branch = g.clone()
print('hash equal before branch:', g.state_hash() == branch.state_hash())
p = g.legal_placements()[0]
print('branch apply:', branch.apply_placement(p.col, p.rot))
print('hash equal after branch :', g.state_hash() == branch.state_hash())

env = TetrisPlacementEnv(seed=42)
obs, info = env.reset()
print('obs keys     :', {k: tuple(v.shape) for k, v in obs.items()})
print('legal actions:', int(info['legal_mask'].sum()), '/ 40')

## 5. 학습 명령 생성

In [ ]:
from pathlib import Path

def command_for(algo: str, run_name: str, preset: str) -> str:
    smoke = preset == 'smoke'
    out = f'checkpoints/{run_name}.pt'

    if algo == 'ppo':
        steps = 4096 if smoke else 1000000
        rollout = 512 if smoke else 2048
        return f'python -m train.ppo_tetris --steps {steps} --rollout {rollout} --eval-every 1 --eval-episodes 1 --out {out}'
    if algo == 'ppo_sparse':
        steps = 4096 if smoke else 1000000
        rollout = 512 if smoke else 2048
        return f'python -m train.ppo_tetris --steps {steps} --rollout {rollout} --shaping-coef 0 --eval-every 1 --eval-episodes 1 --out {out}'
    if algo in ('dqn', 'ddqn'):
        steps = 4096 if smoke else 500000
        warmup = 512 if smoke else 10000
        batch = 64 if smoke else 256
        eval_every = 2048 if smoke else 25000
        return f'python -m train.dqn_tetris --target-mode {algo} --steps {steps} --warmup {warmup} --batch {batch} --eval-every {eval_every} --eval-episodes 1 --out {out}'
    if algo == 'cbmpi':
        iters = 2 if smoke else 20
        states = 512 if smoke else 20000
        epochs = 1 if smoke else 3
        return f'python -m train.cbmpi_tetris --iterations {iters} --states-per-iter {states} --epochs {epochs} --batch 64 --eval-episodes 1 --out {out}'
    if algo == 'cbmpi_value':
        iters = 2 if smoke else 20
        states = 512 if smoke else 20000
        epochs = 1 if smoke else 3
        return f'python -m train.cbmpi_tetris --iterations {iters} --states-per-iter {states} --epochs {epochs} --batch 64 --value-weight 0.25 --eval-episodes 1 --out {out}'
    if algo in ('reinforce', 'a2c', 'nstep_ac'):
        mode = 'nstep-ac' if algo == 'nstep_ac' else algo
        steps = 4096 if smoke else 500000
        rollout = 256 if algo == 'nstep_ac' else 512
        return f'python -m train.policy_gradient_tetris --algo {mode} --steps {steps} --rollout {rollout} --eval-episodes 1 --out {out}'
    if algo == 'cem':
        iters = 2 if smoke else 50
        eps = 8 if smoke else 64
        pieces = 200 if smoke else 2000
        epochs = 1 if smoke else 3
        return f'python -m train.cem_tetris --iterations {iters} --episodes-per-iter {eps} --max-pieces {pieces} --epochs {epochs} --out {out}'
    if algo == 'muzero':
        episodes = 4 if smoke else 500
        pieces = 100 if smoke else 1000
        sims = 4 if smoke else 32
        warmup = 16 if smoke else 2000
        batch = 16 if smoke else 256
        train_steps = 2 if smoke else 32
        distill = 20 if smoke else 2000
        return f'python -m train.muzero_tetris --episodes {episodes} --max-pieces {pieces} --mcts-simulations {sims} --warmup {warmup} --batch {batch} --train-steps-per-episode {train_steps} --distill-steps {distill} --distill-batch {batch} --out {out}'
    raise ValueError(f'unknown ALGO: {algo}')

train_cmd = command_for(ALGO, RUN_NAME, TRAIN_PRESET)
print(train_cmd)

## 6. 학습 실행

처음에는 반드시 `TRAIN_PRESET = 'smoke'`로 한 번 통과시킨 뒤, `long`으로 바꿔 학습하세요.

In [ ]:
import os
import subprocess

os.chdir(f'{REPO_DIR}/python')
subprocess.run(train_cmd, shell=True, check=True)

## 7. ONNX export

MuZero-style은 native checkpoint가 아니라 distill된 `*.policy.pt`를 export합니다. 나머지는 `*.eval_best.pt`가 있으면 그 파일을, 없으면 최신 `.pt`를 export합니다.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

py_dir = Path(REPO_DIR) / 'python'
os.chdir(py_dir)

ckpt_latest = py_dir / 'checkpoints' / f'{RUN_NAME}.pt'
ckpt_eval = py_dir / 'checkpoints' / f'{RUN_NAME}.eval_best.pt'
ckpt_policy = py_dir / 'checkpoints' / f'{RUN_NAME}.policy.pt'

if ALGO == 'muzero':
    ckpt = ckpt_policy
elif ckpt_eval.exists():
    ckpt = ckpt_eval
else:
    ckpt = ckpt_latest

if not ckpt.exists():
    raise FileNotFoundError(f'checkpoint not found: {ckpt}')

out = Path(REPO_DIR) / 'model' / 'bots' / f'{RUN_NAME}.onnx'
out.parent.mkdir(parents=True, exist_ok=True)

print('checkpoint:', ckpt)
print('onnx      :', out)
cmd = [sys.executable, '-m', 'netbot.export_onnx', str(ckpt), str(out)]
print('command   :', ' '.join(cmd))
result = subprocess.run(cmd, cwd=py_dir, text=True, capture_output=True)
if result.stdout:
    print(result.stdout)
if result.stderr:
    print(result.stderr)
result.check_returncode()

## 8. 봇 roster 설정 파일 생성

속도는 `input_interval_ticks`로 지정합니다. 배포 빌드에서는 이 설정값만 사용하고, 디버그 빌드에서만 임시 단축키 조절이 보입니다.

In [ ]:
from pathlib import Path

cfg = Path('../model/bots.cfg')
line = f'model/bots/{RUN_NAME}.onnx|{RUN_NAME}|2\n'
existing = cfg.read_text() if cfg.exists() else ''
if line not in existing:
    cfg.write_text(existing + line)
print(cfg.read_text())

## 9. 다운로드

로컬 게임에는 `.onnx`와 선택적으로 `bots.cfg`만 가져가면 됩니다.

In [ ]:
from google.colab import files

files.download(f'{REPO_DIR}/model/bots/{RUN_NAME}.onnx')
files.download(f'{REPO_DIR}/model/bots.cfg')